# Baseline Experiment Table

This notebook builds a baseline LaTeX table (via `great_tables`) from the
`exp02_heatmap_scan.py` results. Update the parameters in the next cell to
regenerate the table for other settings.


In [10]:
from pathlib import Path
import sys
import re
import json

import numpy as np
import pandas as pd
import great_tables as gt

# === Parameters (edit as needed) ===
RESULTS_DIR_NAME = "result_K30_qtable"
NASH_SUMMARY_NAME = "nash_equilibrium_summary_result_K30_qtable.csv"
MU_BASELINE = 0.25
K_BASELINE = 30
N_VALUES = [2, 3, 5, 7, 10]


In [11]:
def find_pricing_marl_root(start: Path) -> Path:
    # Return the pricing_marl project root (.../pricing_marl).
    for p in [start] + list(start.parents):
        if (p / "pricing_marl" / "data").exists():
            return p / "pricing_marl"
        if p.name == "pricing_marl" and (p / "data").exists():
            return p
    raise FileNotFoundError("Could not locate pricing_marl root from: " + str(start))


def find_experiment_dir(results_dir: Path, label: str, mu: float, k: float) -> Path:
    # Locate experiment folder like scan_{label}_mu{mu}_k{k}.
    target = results_dir / f"scan_{label}_mu{mu}_k{k}"
    if target.exists():
        return target

    pattern = re.compile(r"^scan_(?P<label>.+?)_mu(?P<mu>[-0-9.]+)_k(?P<k>[-0-9.]+)$")
    for d in results_dir.iterdir():
        if not d.is_dir():
            continue
        m = pattern.match(d.name)
        if not m:
            continue
        if m.group("label") != label:
            continue
        if abs(float(m.group("mu")) - float(mu)) < 1e-9 and abs(float(m.group("k")) - float(k)) < 1e-9:
            return d

    raise FileNotFoundError(
        f"No experiment folder found for label={label}, mu={mu}, k={k} in {results_dir}"
    )


def load_config_for_n(exp_dir: Path, n_dir: Path, n: int) -> dict:
    # Prefer explicit config in exp_dir, else fallback to any Config_*.json in N dir.
    candidate = exp_dir / f"Config_N_{n}.json"
    if candidate.exists():
        return json.load(open(candidate))

    config_files = list(n_dir.glob("Config_*.json"))
    if config_files:
        return json.load(open(config_files[0]))

    raise FileNotFoundError(f"No config file found for N={n} in {exp_dir}")


In [12]:
# Resolve project paths
project_root = find_pricing_marl_root(Path.cwd())
sys.path.append(str(project_root))

from src.environment import compute_nash_and_monopoly_static
from src.strategies import ID_TO_NAME, ACT_UNDERCUT, ACT_MATCH, ACT_ABOVE, ACT_UNDER_RESET

results_dir = project_root / "data" / RESULTS_DIR_NAME
nash_summary_path = project_root / "analysis" / "tables" / NASH_SUMMARY_NAME

# Define rule sets (match exp02_heatmap_scan.py)
RULESETS = {
    "3strats": [ACT_UNDERCUT, ACT_MATCH, ACT_ABOVE],
    "4strats": [ACT_UNDERCUT, ACT_MATCH, ACT_ABOVE, ACT_UNDER_RESET],
}

RULESET_LABELS = {
    "3strats": "3 Rules",
    "4strats": "4 Rules",
}


In [13]:
def summarize_experiment(exp_dir: Path, rules_label: str, rule_ids: list[int], n_values: list[int]):
    rows = []
    algo_label = rules_label

    for n in n_values:
        n_dir = exp_dir / f"N_{n}"
        if not n_dir.exists():
            continue

        parquet_files = sorted([p for p in n_dir.glob("run_*.parquet") if "_qtable" not in p.name])
        if not parquet_files:
            continue

        cfg = load_config_for_n(exp_dir, n_dir, n)
        p_nash, p_monopoly = compute_nash_and_monopoly_static(
            cfg["num_sellers"],
            cfg["a_val"],
            cfg["mu"],
            cfg["a0"],
            cfg["c_val"],
        )

        deltas = []
        episodes = []
        lowest_price_means = []

        for pf in parquet_files:
            df = pd.read_parquet(pf, columns=["delta", "episode", "price_min"])
            if df.empty:
                continue
            deltas.append(float(df["delta"].mean()))
            lowest_price_means.append(float(df["price_min"].mean()))
            episodes.append(int(df["episode"].iloc[0]))  # first recorded episode = converge

        if not deltas:
            continue

        delta_mean = float(np.mean(deltas))
        if len(deltas) > 1:
            delta_sd = float(np.std(deltas, ddof=1))
        else:
            delta_sd = 0.0

        lowest_price_mean = float(np.mean(lowest_price_means))
        if len(lowest_price_means) > 1:
            lowest_price_sd = float(np.std(lowest_price_means, ddof=1))
        else:
            lowest_price_sd = 0.0

        rows.append(
            {
                "Algorithmic Rules": algo_label,
                "N": n,
                "Delta Mean": delta_mean,
                "Delta SD": delta_sd,
                "Monopoly Price": float(p_monopoly),
                "Bertrand Price": float(p_nash),
                "Lowest Price Mean": lowest_price_mean,
                "Lowest Price SD": lowest_price_sd,
                "Time to Converge": int(round(np.mean(episodes))),
            }
        )

    return rows


In [14]:
def compute_converged_rates(exp_dir: Path, rules_label: str, n_values: list[int]):
    rows = []
    for n in n_values:
        n_dir = exp_dir / f"N_{n}"
        if not n_dir.exists():
            continue

        parquet_files = sorted([p for p in n_dir.glob("run_*.parquet") if "_qtable" not in p.name])
        if not parquet_files:
            continue

        flags = []
        for pf in parquet_files:
            df = pd.read_parquet(pf, columns=["converged"])
            if df.empty:
                continue
            flags.append(int(df["converged"].iloc[0]))

        if not flags:
            continue

        rows.append(
            {
                "Algorithmic Rules": rules_label,
                "N": n,
                "Converged Rate": float(np.mean(flags)),
                "Runs": len(flags),
            }
        )

    return rows


In [15]:
# Build the baseline table for (mu, K)
rows = []
for label, rule_ids in RULESETS.items():
    exp_dir = find_experiment_dir(results_dir, label, MU_BASELINE, K_BASELINE)
    rows.extend(
        summarize_experiment(
            exp_dir=exp_dir,
            rules_label=RULESET_LABELS[label],
            rule_ids=rule_ids,
            n_values=N_VALUES,
        )
    )

baseline_df = pd.DataFrame(rows)

if not nash_summary_path.exists():
    raise FileNotFoundError(
        f"Missing Nash summary CSV: {nash_summary_path}. Run analysis/count_nash_equilibrium_runs.py first."
    )

nash_summary_df = pd.read_csv(nash_summary_path)
nash_summary_df = nash_summary_df[
    (np.isclose(nash_summary_df["mu"], MU_BASELINE)) & (nash_summary_df["K"] == K_BASELINE)
].copy()
nash_summary_df["Algorithmic Rules"] = nash_summary_df["strategy_label"].map(RULESET_LABELS)
nash_summary_df["Nash Equilibrium Rate"] = (
    nash_summary_df["num_NashEqu"] / nash_summary_df["num_runs"]
)
nash_summary_df = nash_summary_df[["Algorithmic Rules", "N", "Nash Equilibrium Rate"]]

baseline_df = baseline_df.merge(
    nash_summary_df,
    on=["Algorithmic Rules", "N"],
    how="left",
    validate="one_to_one",
)

# Order rows for stable display
baseline_df = baseline_df.sort_values(["Algorithmic Rules", "N"], ascending=[True, True]).reset_index(drop=True)

# Merge Algorithmic Rules cells visually by blanking repeated labels
baseline_df["Algorithmic Rules"] = baseline_df["Algorithmic Rules"].mask(
    baseline_df["Algorithmic Rules"].duplicated(), ""
)

# Format Delta and Lowest Price with SD on the next line: mean then (sd)
baseline_df["Delta Mean"] = baseline_df.apply(
    lambda r: f"{r['Delta Mean']:.2f}\n({r['Delta SD']:.2f})",
    axis=1,
)

baseline_df["Lowest Price"] = baseline_df.apply(
    lambda r: f"{r['Lowest Price Mean']:.2f}\n({r['Lowest Price SD']:.2f})",
    axis=1,
)

baseline_df = baseline_df.drop(columns=["Delta SD", "Lowest Price Mean", "Lowest Price SD"])
baseline_df = baseline_df[[
    "Algorithmic Rules",
    "N",
    "Delta Mean",
    "Monopoly Price",
    "Bertrand Price",
    "Lowest Price",
    "Time to Converge",
    "Nash Equilibrium Rate",
]]
for col in ["Monopoly Price", "Bertrand Price"]:
    baseline_df[col] = baseline_df[col].round(2)


# Display raw data for quick debugging
baseline_df


,Algorithmic Rules,N,Delta Mean,Monopoly Price,Bertrand Price,Lowest Price,Time to Converge
0,3 Rules,2,0.98\n(0.01),1.92,1.47,1.97\n(0.02),656848
1,,3,1.00\n(0.00),2.00,1.37,2.00\n(0.00),522653
2,,5,0.98\n(0.02),2.10,1.31,2.17\n(0.05),703546
3,,7,0.96\n(0.01),2.16,1.29,2.28\n(0.03),774200
4,,10,0.95\n(0.05),2.23,1.28,2.36\n(0.06),795190
5,4 Rules,2,0.98\n(0.01),1.92,1.47,1.96\n(0.03),647099
6,,3,1.00\n(0.01),2.00,1.37,2.00\n(0.02),538904
7,,5,0.96\n(0.00),2.10,1.31,2.21\n(0.01),755862
8,,7,0.96\n(0.00),2.16,1.29,2.29\n(0.00),766718
9,,10,0.90\n(0.16),2.23,1.28,2.30\n(0.19),745030


In [16]:
# Convergence check (should be all 1.0)
converged_rows = []
for label in RULESETS:
    exp_dir = find_experiment_dir(results_dir, label, MU_BASELINE, K_BASELINE)
    converged_rows.extend(
        compute_converged_rates(
            exp_dir=exp_dir,
            rules_label=RULESET_LABELS[label],
            n_values=N_VALUES,
        )
    )

converged_df = pd.DataFrame(converged_rows)
converged_df = converged_df.sort_values(["Algorithmic Rules", "N"], ascending=[True, True]).reset_index(drop=True)
converged_df["Converged Rate"] = converged_df["Converged Rate"].round(2)

converged_df


,Algorithmic Rules,N,Converged Rate,Runs
0,3 Rules,2,1.0,100
1,3 Rules,3,1.0,100
2,3 Rules,5,1.0,100
3,3 Rules,7,1.0,100
4,3 Rules,10,1.0,100
5,4 Rules,2,1.0,100
6,4 Rules,3,1.0,100
7,4 Rules,5,1.0,100
8,4 Rules,7,1.0,100
9,4 Rules,10,1.0,100


In [17]:
# Build the GT table
table = (
    gt.GT(baseline_df)
    .cols_label(
        **{
            "Algorithmic Rules": "Algorithmic Rules",
            "N": "N",
            "Delta Mean": r"$\Delta$",
            "Monopoly Price": r"$p^M$",
            "Bertrand Price": r"$p^N$",
            "Lowest Price": "Lowest Price",
            "Time to Converge": "Time to Converge",
            "Nash Equilibrium Rate": "Nash Eq.",
        }
    )
    .fmt_number(columns=["Monopoly Price", "Bertrand Price"], decimals=2)
    .fmt_percent(columns=["Nash Equilibrium Rate"], decimals=1)
    .fmt_integer(columns=["N", "Time to Converge"])
)

# Render in notebook
table


Algorithmic Rules,N,$\Delta$,$p^M$,$p^N$,Lowest Price,Time to Converge
3 Rules,2,0.98 (0.01),1.92,1.47,1.97 (0.02),"656,848"
,3,1.00 (0.00),2.00,1.37,2.00 (0.00),"522,653"
,5,0.98 (0.02),2.10,1.31,2.17 (0.05),"703,546"
,7,0.96 (0.01),2.16,1.29,2.28 (0.03),"774,200"
,10,0.95 (0.05),2.23,1.28,2.36 (0.06),"795,190"
4 Rules,2,0.98 (0.01),1.92,1.47,1.96 (0.03),"647,099"
,3,1.00 (0.01),2.00,1.37,2.00 (0.02),"538,904"
,5,0.96 (0.00),2.10,1.31,2.21 (0.01),"755,862"
,7,0.96 (0.00),2.16,1.29,2.29 (0.00),"766,718"
,10,0.90 (0.16),2.23,1.28,2.30 (0.19),"745,030"


In [18]:
# Export LaTeX
output_dir = project_root / "analysis" / "tables"
output_dir.mkdir(parents=True, exist_ok=True)
output_path = output_dir / f"baseline_mu{MU_BASELINE}_k{K_BASELINE}.tex"

latex_str = table.as_latex()
latex_str = latex_str.replace(r'\$\\Delta\$', r'$\Delta$')
latex_str = latex_str.replace(r'\$p\^M\$', r'$p^M$')
latex_str = latex_str.replace(r'\$p\^N\$', r'$p^N$')

# Insert a separator between the 3- and 4-rule blocks.
latex_str = latex_str.replace(
    "\n4 Rules:",
    "\n\\midrule\\addlinespace[2.5pt]\n4 Rules:",
)

# Strip the outer table wrapper and caption so this file can be \input{}-ed.
latex_str = re.sub(r"\\begin\{table\}\[!t\]\s*", "", latex_str)
latex_str = re.sub(r"\\end\{table\}\s*", "", latex_str)
latex_str = re.sub(r"\\caption\\*?\\{[\\s\\S]*?\\}\\s*", "", latex_str, flags=re.S)
latex_str = re.sub(r"\n{3,}", "\n\n", latex_str).strip() + "\n"

output_path.write_text(latex_str)

print(f"Wrote LaTeX to: {output_path}")
print("\n--- LaTeX Preview ---\n")
print(latex_str)

Wrote LaTeX to: /Users/liushijian/Documents/GitHub/Amazon_BuyBox_Reinforcement_Learning/pricing_marl/analysis/tables/baseline_mu0.25_k30.tex

--- LaTeX Preview ---

\fontsize{12.0pt}{14.4pt}\selectfont

\begin{tabular*}{\linewidth}{@{\extracolsep{\fill}}lrlrrlr}
\toprule
Algorithmic Rules & N & $\Delta$ & $p^M$ & $p^N$ & Lowest Price & Time to Converge \\ 
\midrule\addlinespace[2.5pt]
3 Rules & 2 & 0.98
(0.01) & 1.92 & 1.47 & 1.97
(0.02) & 656,848 \\
 & 3 & 1.00
(0.00) & 2.00 & 1.37 & 2.00
(0.00) & 522,653 \\
 & 5 & 0.98
(0.02) & 2.10 & 1.31 & 2.17
(0.05) & 703,546 \\
 & 7 & 0.96
(0.01) & 2.16 & 1.29 & 2.28
(0.03) & 774,200 \\
 & 10 & 0.95
(0.05) & 2.23 & 1.28 & 2.36
(0.06) & 795,190 \\
4 Rules & 2 & 0.98
(0.01) & 1.92 & 1.47 & 1.96
(0.03) & 647,099 \\
 & 3 & 1.00
(0.01) & 2.00 & 1.37 & 2.00
(0.02) & 538,904 \\
 & 5 & 0.96
(0.00) & 2.10 & 1.31 & 2.21
(0.01) & 755,862 \\
 & 7 & 0.96
(0.00) & 2.16 & 1.29 & 2.29
(0.00) & 766,718 \\
 & 10 & 0.90
(0.16) & 2.23 & 1.28 & 2.30
(0.19) & 745,030